# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset using the `mlcroissant` library, following a standardized approach for Croissant-style datasets.

### Dataset Source
The dataset is defined via a Croissant schema JSON-LD file, making it machine-actionable for data loading and processing.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
# Print dataset summary using the metadata attributes
md = dataset.metadata
print(f"Name: {md.name}\nDescription: {md.description}\nVersion: {md.version}\nLicense: {md.license}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we identify all record sets by their `@id`, and within each record set, list all available fields (columns) by `@id` and field name.

In [ ]:
print("Available Record Sets and Fields:")

record_sets = list(dataset.metadata.record_sets)
records_overview = {}
for rs in record_sets:
    print(f"- Record Set @id: {rs.id}")
    fields = rs.fields
    records_overview[rs.id] = [f.id for f in fields]
    for f in fields:
        print(f"    - Field @id: {f.id:35s} name: {getattr(f, 'name', f.id)}")
if not record_sets:
    print("No record sets detected in main metadata; attempting to enumerate data files (distributions)...")
    # Show distributions if available
    dists = dataset.metadata.distributions if hasattr(dataset.metadata, 'distributions') else []
    for dist in dists:
        print(f"  Distribution @id: {dist.id}, url: {dist.content_url}, encoding: {dist.encoding_format}")

## 3. Data Extraction
We're going to load data from a selected record set into a pandas DataFrame for downstream analysis.

Each table is referenced by its record set `@id` and each field by its unique `@id`. Edit the example as needed if inspecting additional record sets.

In [ ]:
# Gather all record set @id strings
record_sets = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}
print(f"Record sets in the package: {record_sets}")

# Example: Load all records from each record set into a DataFrame
for record_set_id in record_sets:
    recs = list(dataset.records(record_set=record_set_id))
    if len(recs) > 0:
        df = pd.DataFrame(recs)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for {record_set_id} with shape {df.shape}")
    else:
        print(f"No records returned for {record_set_id}")

# Pick the first table if available and show column ids
if record_sets:
    first_rs = record_sets[0]
    if first_rs in dataframes:
        print('Sample columns:', dataframes[first_rs].columns.tolist())
        display(dataframes[first_rs].head(3))

## 4. Exploratory Data Analysis (EDA)
Let us perform basic analysis on a numeric column (e.g., `cr:field/mw` for molecular weight if available) using its Croissant `@id`.

We'll filter for proteins with molecular weight > 10000 Da, normalize this column, and if appropriate, group by accession (`cr:field/accession`).

In [ ]:
# Please change these @id's for your dataset, as discovered in the data overview step.
# We'll use typical probable @id's. If mw (molecular weight) is available, use that.

# Set these based on cell 5 output, e.g.:
record_set_id = record_sets[0] if record_sets else None
numeric_field_id = None  # will be populated below
group_field_id = None

if record_set_id is not None and record_set_id in dataframes:
    df = dataframes[record_set_id]
    # Suggest likely fields for numeric analysis
    candidates = [col for col in df.columns if 'mw' in col.lower() or 'weight' in col.lower()]
    if not candidates:
        candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if candidates:
        numeric_field_id = candidates[0]
    else:
        print('No suitable numeric field detected, skipping EDA example.')
    
    # Try to group by 'accession' if present
    group_candidates = [c for c in df.columns if 'accession' in c.lower() or 'id' in c.lower()]
    if group_candidates:
        group_field_id = group_candidates[0]

    if numeric_field_id:
        print(f"Analyzing numeric field: {numeric_field_id}")
        # Remove records with missing or zero molecular weight
        threshold = 10000
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records with {numeric_field_id} > {threshold}.")

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"First few normalized values:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by accession (if available)
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped average {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
else:
    print('No available DataFrame for EDA.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field (e.g., molecular weight) if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and record_set_id in dataframes and numeric_field_id:
    df = dataframes[record_set_id]
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id].dropna(), bins=40, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated:
- How to load and inspect Croissant metadata and records using the `mlcroissant` library,
- How to reference all dataset components using their unique `@id`,
- How to extract records into pandas DataFrames using their record set `@id`,
- Basic exploratory analysis and normalization using field `@id`s,
- A starter visualization to explore numeric distributions.

You can now proceed with further protein expression analysis, feature engineering, or integrate this workflow into your mass spectrometry bioinformatics toolkit!